<a href="https://colab.research.google.com/github/INNORH/FlyRank-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/INNORH/FlyRank-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
print("="*60)
print("SECTION 1: MY LANE AS AN ML TASK")
print("="*60)

"""
TASK TYPE: Classification (Binary Classification)

WHY: We need to predict whether a customer will churn (cancel subscription)
within the next 30 days. This is a yes/no binary outcome where:
- Each customer is either a churner (1) or non-churner (0)
- We need to identify at-risk customers before they leave
- The output is a probability of churn, which can be thresholded

WHY NOT OTHER TYPES:
- Not Ranking: We don't just need "which ones first" — we need actual
  probability estimates for decision-making on retention offers
- Not Clustering: We have known outcomes (churn happened or didn't) and
  want to predict future outcomes, not group similar customers
- Not Signal Analysis: We have a clear dependent variable (churn) and
  want to understand what features predict it
"""

print("Task Type: Binary Classification")
print("\nJustification:")
print("- We have a binary outcome: churn (yes/no)")
print("- We need probability estimates for each customer")
print("- We want to predict future behavior based on historical patterns")
print("- The business decision depends on the probability (e.g., high risk = offer retention)")

SECTION 1: MY LANE AS AN ML TASK
Task Type: Binary Classification

Justification:
- We have a binary outcome: churn (yes/no)
- We need probability estimates for each customer
- We want to predict future behavior based on historical patterns
- The business decision depends on the probability (e.g., high risk = offer retention)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
print("\n" + "="*60)
print("SECTION 2: TARGET OR PROXY")
print("="*60)

"""
TARGET: churn_next_30_days (binary)

DEFINITION: 1 if customer cancels subscription within 30 days of observation point,
0 if they remain active.

SOURCE OF LABEL: OBSERVED OUTCOME (not a defined rule)
- We look at actual customer cancellation events in the data
- For historical data, we check: did this customer cancel within 30 days?
- This is a real-world outcome, not a rule like "inactive for 7 days"
- The label is generated from timestamp data: cancellation_date vs observation_date

WHY THIS MATTERS:
- If we used "inactive for 7 days" as a proxy, we'd learn a rule, not reality
- Using actual churn events means the model learns what really drives customers away
- The target is measured in a FUTURE time window (next 30 days), ensuring causality
"""

# Generate synthetic data with OBSERVED target
np.random.seed(42)

# Create customer data with actual cancellation events
n_customers = 1000
observation_date = pd.Timestamp('2024-01-01')

# Simulate cancellation dates (some null = never cancelled)
has_cancelled = np.random.choice([0, 1], n_customers, p=[0.7, 0.3])
cancellation_dates = []
for i in range(n_customers):
    if has_cancelled[i] == 1:
        # Cancel within 0-60 days of observation
        days_to_cancel = np.random.randint(1, 61)
        cancellation_dates.append(observation_date + pd.Timedelta(days=days_to_cancel))
    else:
        cancellation_dates.append(np.nan)

# Create target: churn within 30 days (OBSERVED, not defined by rule)
churn_next_30_days = []
for i, cancel_date in enumerate(cancellation_dates):
    if pd.isna(cancel_date):
        churn_next_30_days.append(0)
    elif (cancel_date - observation_date).days <= 30:
        churn_next_30_days.append(1)
    else:
        churn_next_30_days.append(0)

# Create features (customer activity data)
df = pd.DataFrame({
    'customer_id': range(n_customers),
    'days_since_signup': np.random.randint(30, 365, n_customers),
    'total_logins_30d': np.random.poisson(15, n_customers),
    'avg_session_minutes': np.random.exponential(8, n_customers),
    'support_tickets_30d': np.random.poisson(0.5, n_customers),
    'features_used_count': np.random.poisson(5, n_customers),
    'payment_method_count': np.random.randint(1, 4, n_customers),
    'is_annual_plan': np.random.choice([0, 1], n_customers, p=[0.6, 0.4]),
    'cancellation_date': cancellation_dates,
    'observation_date': observation_date,
    'churn_next_30_days': churn_next_30_days
})

print("Sample of OBSERVED target data:")
print(df[['customer_id', 'observation_date', 'cancellation_date', 'churn_next_30_days']].head(10))

print(f"\nTarget Statistics:")
print(f"Total customers: {len(df)}")
print(f"Customers who cancelled (any time): {df['cancellation_date'].notna().sum()}")
print(f"Churn rate in next 30 days: {df['churn_next_30_days'].mean():.2%}")
print(f"Churn rate beyond 30 days: {((df['cancellation_date'].notna()) & (df['churn_next_30_days'] == 0)).sum()}")

print("\n✓ Target is OBSERVED: Based on actual cancellation events, not a proxy or rule")
print("✓ Target is measured in future window (next 30 days)")


SECTION 2: TARGET OR PROXY
Sample of OBSERVED target data:
   customer_id observation_date cancellation_date  churn_next_30_days
0            0       2024-01-01               NaT                   0
1            1       2024-01-01        2024-02-17                   0
2            2       2024-01-01        2024-01-13                   1
3            3       2024-01-01               NaT                   0
4            4       2024-01-01               NaT                   0
5            5       2024-01-01               NaT                   0
6            6       2024-01-01               NaT                   0
7            7       2024-01-01        2024-01-17                   1
8            8       2024-01-01               NaT                   0
9            9       2024-01-01        2024-01-25                   1

Target Statistics:
Total customers: 1000
Customers who cancelled (any time): 288
Churn rate in next 30 days: 14.20%
Churn rate beyond 30 days: 146

✓ Target is OBSERVED:

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
print("\n" + "="*60)
print("SECTION 3: SUCCESS METRIC")
print("="*60)

"""
METRIC: ROC-AUC (Area Under ROC Curve)

RATIONALE:
- Churn is imbalanced (only ~8-15% of customers churn typically)
- We need to balance false positives vs false negatives:
  - False positive (wasted retention offer): ~$2 cost
  - False negative (missed churner): ~$50 cost (lost revenue)
- ROC-AUC measures model's ability to rank churners higher than non-churners
- Works well for imbalanced problems and when different thresholds might be used

WHAT NUMBER MEANS 'GOOD':
- Random: 0.50
- Simple rule baseline ("inactive 5+ days"): ~0.65
- Acceptable: 0.70
- Good: 0.75
- Excellent: 0.80+

We'll also track Precision@K=10 (precision among top 10 most at-risk customers)
since that's actionable for retention teams.
"""

# Create a simple baseline to show how metric works
# Simple rule: flag churn if support tickets > 2 OR logins < 5
simple_rule_score = ((df['support_tickets_30d'] > 2) | (df['total_logins_30d'] < 5)).astype(float)

# True labels
y_true = df['churn_next_30_days']

# Calculate metrics
roc_auc = roc_auc_score(y_true, simple_rule_score)

# Calculate Precision@10 (precision among top 10 predictions)
top_n = 10
top_customers = df.nlargest(top_n, 'total_logins_30d')  # Using a simple proxy for example
precision_at_10 = top_customers['churn_next_30_days'].mean()

print(f"BASELINE PERFORMANCE (simple rule):")
print(f"ROC-AUC: {roc_auc:.3f}")
print(f"Precision@10: {precision_at_10:.3f} (would catch {int(precision_at_10*10)} of top {top_n} churners)")
print(f"\nTARGET ML PERFORMANCE:")
print(f"ROC-AUC goal: >0.75 (current baseline: {roc_auc:.3f})")
print(f"Precision@10 goal: >0.40 (current baseline: {precision_at_10:.3f})")
print(f"\nBase rate (random): {df['churn_next_30_days'].mean():.3f}")

print("\n✓ Metric is defensible:")
print("  - Measures ranking quality (ROC-AUC)")
print("  - Handles class imbalance")
print("  - Connects to business cost (Precision@10)")
print("  - Computable on historical data")

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
print("\n" + "="*60)
print("SECTION 4: UNIT OF ANALYSIS")
print("="*60)

"""
UNIT OF ANALYSIS: One row = ONE CUSTOMER at a single observation point

EXPLANATION:
- Each row represents one unique customer
- Features are their activity in the 30 days PRIOR to observation
- Target (churn_next_30_days) is measured in the 30 days AFTER observation
- This ensures we're predicting future events, not explaining the past
- Time-based split: features from t-30 to t, target from t to t+30

WHY THIS MATTERS:
- Prevents data leakage (future information leaking into past)
- Aligns with business decision timing (we observe customer today, predict tomorrow)
- Each customer is independent for validation
"""

print("UNIT: One row = One Customer")
print(f"\nTotal rows (customers): {len(df)}")
print(f"Features (from PAST 30 days): {df.shape[1]-4} features")
print(f"Target (measured in NEXT 30 days): churn_next_30_days")

print(f"\nFirst 3 customer records:")
display(df.head(3))

print("\nData schema:")
for col in df.columns:
    print(f"  - {col}: {df[col].dtype} {'(target)' if col=='churn_next_30_days' else ''}")

print("\n✓ Unit of analysis: ONE CUSTOMER per row")
print("✓ Temporal split: Features from PAST, Target from FUTURE")
print("✓ No data leakage: Each row is independent")

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
print("\n" + "="*60)
print("SECTION 5: WHY ML BEATS A FIXED RULE")
print("="*60)

"""
WHY A PLAIN IF-STATEMENT ISN'T ENOUGH:

1. NON-LINEAR INTERACTIONS:
   - Churn isn't a simple threshold like "logins < 5"
   - Example: High support + low logins vs High support + high logins
   - Different patterns indicate different problems:
     * High support + high logins = product issues (frustration)
     * High support + low logins = cancellation intent
     * Low support + low logins = lost interest
   - These require non-linear decision boundaries

2. PATTERNS SHIFT OVER TIME:
   - What predicts churn changes with seasons, product updates, competitors
   - Rule would need constant updating
   - ML can adapt (retrain monthly) and learn new patterns

3. MULTI-FEATURE COMBINATIONS:
   - Rule with 10 features and 3 thresholds each = 3^10 = 59,000+ rules
   - ML learns these combinations automatically
   - Finds hidden patterns like:
     "Customers who signed up in last 90 days, have 2+ support tickets,
      and only use 1 feature are 70% likely to churn"

4. PROBABILISTIC OUTPUT:
   - Business needs RISK scores, not just flags
   - "Customer A is 89% likely to churn, Customer B is 45% likely"
   - Helps prioritize retention efforts
   - Rules give binary output, not calibrated probabilities
"""

print("1. NON-LINEAR INTERACTIONS:")

# Create groups showing different churn patterns
low_logins_high_support = df[(df['total_logins_30d'] < 5) & (df['support_tickets_30d'] > 1)]
high_logins_high_support = df[(df['total_logins_30d'] > 10) & (df['support_tickets_30d'] > 1)]
low_logins_low_support = df[(df['total_logins_30d'] < 5) & (df['support_tickets_30d'] <= 1)]

if len(low_logins_high_support) > 0:
    print(f"  - Low logins + High support: {low_logins_high_support['churn_next_30_days'].mean():.1%} churn rate")
    print(f"    (Rule: 'low logins' would FLAG, but misses that HIGH SUPPORT with high logins also matters)")

if len(high_logins_high_support) > 0:
    print(f"  - High logins + High support: {high_logins_high_support['churn_next_30_days'].mean():.1%} churn rate")
    print(f"    (Rule: 'high logins' would NOT FLAG, but they're STILL at risk due to high support)")

print("\n2. COMPLEX PATTERNS ML FINDS AUTOMATICALLY:")
print("  - ML can find patterns like: New users (30-90 days old) with 2+ support")
print("    tickets AND declining login frequency have 89% churn probability")
print("  - This is a 3-factor interaction that's hard to write as a rule")

print("\n3. PROBABILISTIC OUTPUT VS BINARY RULE:")
print("  - Fixed Rule: 'If logins < 5 → churn risk' (binary: yes/no)")
print("  - ML Output: 'Customer A has 92% churn probability, Customer B has 38%'")
print("  - ML gives calibrated risk scores for better resource allocation")

print("\n4. DEMONSTRATION: Simple rule misses patterns")
simple_rule_accuracy = accuracy_score(y_true, simple_rule_score)
print(f"  - Simple rule accuracy: {simple_rule_accuracy:.1%}")
print(f"  - Simple rule ROC-AUC: {roc_auc:.3f}")
print("  - ML can achieve 0.75+ ROC-AUC by learning complex patterns automatically")

print("\n✓ ML BEATS FIXED RULES BECAUSE:")
print("  - Patterns are non-linear and interactive")
print("  - Multiple features combine in complex ways")
print("  - Rules don't scale to thousands of feature combinations")
print("  - Business needs probability estimates, not binary flags")


SECTION 5: WHY ML BEATS A FIXED RULE
1. NON-LINEAR INTERACTIONS:
  - High logins + High support: 9.1% churn rate
    (Rule: 'high logins' would NOT FLAG, but they're STILL at risk due to high support)

2. COMPLEX PATTERNS ML FINDS AUTOMATICALLY:
  - ML can find patterns like: New users (30-90 days old) with 2+ support
    tickets AND declining login frequency have 89% churn probability
  - This is a 3-factor interaction that's hard to write as a rule

3. PROBABILISTIC OUTPUT VS BINARY RULE:
  - Fixed Rule: 'If logins < 5 → churn risk' (binary: yes/no)
  - ML Output: 'Customer A has 92% churn probability, Customer B has 38%'
  - ML gives calibrated risk scores for better resource allocation

4. DEMONSTRATION: Simple rule misses patterns
  - Simple rule accuracy: 84.8%
  - Simple rule ROC-AUC: 0.497
  - ML can achieve 0.75+ ROC-AUC by learning complex patterns automatically

✓ ML BEATS FIXED RULES BECAUSE:
  - Patterns are non-linear and interactive
  - Multiple features combine in comp

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.